# Magic Eraser in Colab: all versions + demo

Runtime → Change runtime type → **T4 GPU** before starting.

This notebook only calls scripts in `src/`. Put new logic in `src/`, not here.
Sections 6–9 are independent: run only the versions you need. Each version takes roughly 5–15 min on a T4.

In [ ]:
!nvidia-smi

## 1. Get the code (with the ProPainter submodule)

In [ ]:
!git clone --recurse-submodules https://github.com/Norrozo/5810finalproject.git
%cd 5810finalproject

## 2. Install the main environment (SAM 2, Grounding DINO, Gradio)
SAM 2 takes a few minutes to install.

In [ ]:
!pip install -q -r requirements.txt transformers gradio
!pip install -q "git+https://github.com/facebookresearch/sam2.git"

## 3. Install ProPainter in its own environment (only for I4 and the demo's ProPainter option)
ProPainter's dependencies can conflict with SAM 2's, so it gets a separate virtualenv. The pipeline only talks to it through files.
`src/inpaint/propainter.py` looks for `.venv-propainter/bin/python`, so we link it there.

In [ ]:
!pip install -q virtualenv
!virtualenv -q /content/pp_env
!/content/pp_env/bin/pip install -q -r envs/propainter.txt
!ln -sfn /content/pp_env .venv-propainter
!.venv-propainter/bin/python -c "import torch; print('ProPainter env OK, CUDA:', torch.cuda.is_available())"

## 4. (Optional) Keep outputs on Google Drive
Colab wipes files when the session ends. Linking `runs/` to Drive keeps them, and interrupted runs resume where they stopped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/magic-eraser/runs
!rm -rf runs && ln -s /content/drive/MyDrive/magic-eraser/runs runs

## 5. Download DAVIS 2017 and build the benchmark (test + dev sets)
About 800 MB. If the link has moved, get "TrainVal - Images and Annotations (480p)" from davischallenge.org.

In [ ]:
!wget -q https://data.vision.ee.ethz.ch/csergi/share/davis/DAVIS-2017-trainval-480p.zip
!unzip -q DAVIS-2017-trainval-480p.zip && rm DAVIS-2017-trainval-480p.zip
!python -m src.make_synthetic --config configs/B0.yaml --davis-root DAVIS
!python -m src.make_synthetic --config configs/B0.yaml --davis-root DAVIS --clips-file configs/dev_clips.txt

In [ ]:
!python -m tests.smoke_test

## 6. B0 baseline (run first: I1, I3, I3b, I4 reuse its SAM 2 masks)

In [ ]:
!python -m src.run_pipeline --config configs/B0.yaml

## 7. Iterations that only change later stages (fast: reuse B0's masks)

In [ ]:
!python -m src.run_pipeline --config configs/I1.yaml   # mask refinement
!python -m src.run_pipeline --config configs/I3.yaml   # our flow-guided inpainting
!python -m src.run_pipeline --config configs/I3b.yaml  # + hop cap (5)

## 8. I4: ProPainter (needs section 3)
`configs/I4.yaml` uses `subvideo_length: 20` and `fp16` for a 16 GB laptop. They also work on a T4.

In [ ]:
!python -m src.run_pipeline --config configs/I4.yaml

## 9. I2: text prompts (re-runs SAM 2 from a Grounding DINO box)

In [ ]:
!python -m src.run_pipeline --config configs/I2.yaml

## 10. Results

In [ ]:
!python -m src.summarize

In [ ]:
from IPython.display import Image, display
for v in ['B0', 'I3b', 'I4']:
    print(v); display(Image(filename=f'runs/{v}/dog_on_soapbox/stills.png', width=900))

## 11. (Optional) Re-run the dev-set tuning for the hop cap
Writes to `results/tuning.csv`, never `results/results.csv`.

In [ ]:
# !python -m src.sweep --config configs/I3.yaml --key inpaint.max_hops --values 1 2 3 5 8 none

## 12. Your own video, by text prompt

In [ ]:
# Upload a short clip with the file browser on the left, then:
# !python -m src.run_pipeline --config configs/showcase.yaml --video my_clip.mp4 --name my_clip --text "remove the dog"
# Output: runs/showcase/my_clip/side_by_side.mp4

## 13. Demo with a public link
Opens a Gradio page you can share for 72 hours. Stop the cell to shut it down.

In [ ]:
!python -m src.demo --share

## 14. Save results
`results/results.csv` is small. Download it, or commit it to the repo after each run.

In [ ]:
from google.colab import files
files.download('results/results.csv')